In [ ]:
path_to_metrics=""
save_directory=""

In [ ]:
# ===== BOXPLOT with FDR correction =====
from statsmodels.stats.multitest import fdrcorrection
import matplotlib as mpl
import matplotlib.pyplot as plt
import os
import pandas as pd
import seaborn as sns
import numpy as np
from scipy.stats import ttest_rel, wilcoxon, shapiro
mpl.rcParams["font.family"] = "Arial"
plt.style.use("seaborn-v0_8-whitegrid")
font_to_use = "Arial"
df = pd.read_csv(os.path.join(path_to_metrics, "connectome_no_density_metrics_global.csv"))

all_results = []  # lista con (metric, technique, p_raw, pos, test_type)
output_dir = os.path.join(save_directory, "boxplots")
os.makedirs(output_dir, exist_ok=True)
# List of metrics to plot
metrics = [
    "sparsity", "mean_connectivity", "assortativity", "avg_clustering",
    "global_efficiency", "avg_shortest_path", "transitivity"
]
templates = df["template"].unique()
technique_mapping = {
    "25": "ATR-25-ACT",
    "30": "ATR-30-ACT",
    "35": "ATR-35-ACT",
    "40": "ATR-40-ACT",
    "45": "ATR-45-ACT",
    "50": "ATR-50-ACT",
    "act": "FAST-ACT",
    "no_act": "NOACT"
}
df["technique"] = df["technique"].astype(str).map(technique_mapping).fillna(df["technique"])
techniques = sorted(df["technique"].unique())
# ===== raw p-values =====
for metric in metrics:
    for i, tech in enumerate(techniques):
        group = df[df["technique"] == tech]
        template_data = {}
        subjects = []

        for template in templates:
            template_group = group[group["template"] == template]
            template_data[template] = template_group.set_index("excel_id")[metric]
            subjects.append(set(template_group["excel_id"]))

        common_subjects = list(subjects[0].intersection(subjects[1]))

        if len(common_subjects) > 1:
            data1 = template_data[templates[0]].loc[common_subjects].values
            data2 = template_data[templates[1]].loc[common_subjects].values
            diff = data1 - data2

            _, norm_p = shapiro(diff)
            if norm_p > 0.05:
                _, p_val = ttest_rel(data1, data2)
                test_type = "t-test appaiato"
            else:
                _, p_val = wilcoxon(data1, data2)
                test_type = "Wilcoxon"

            print(f"Usando {test_type} per {tech} su {metric} con p_raw={p_val:.4f}")
            all_results.append((metric, tech, p_val, i, test_type))

# ===== Global FDR correction =====
raw_pvals = [p for (_, _, p, _, _) in all_results]
reject, pvals_corrected = fdrcorrection(raw_pvals, alpha=0.05, method='indep')

corrected_dict = {}
for (m, t, p_raw, pos, test_type), p_corr, rej in zip(all_results, pvals_corrected, reject):
    corrected_dict[(m, t)] = (p_raw, p_corr, rej, pos, test_type)

df_pvalues = pd.DataFrame([
    (m, t, test_type, p_raw, p_corr, rej, pos)
    for (m, t, p_raw, pos, test_type), p_corr, rej in zip(all_results, pvals_corrected, reject)
], columns=["metric", "technique", "test_type", "p_raw", "p_corr", "significant", "pos"])

df_pvalues.to_csv(os.path.join(output_dir, "pvalues_corrected_global.csv"), index=False)

# ===== Plotting =====
for metric in metrics:
    plt.figure(figsize=(10, 6))
    ax = sns.boxplot(
        data=df,
        x="technique",
        y=metric,
        hue="template",
        palette="rocket",
        showfliers=False
    )

    y_min, y_max = ax.get_ylim()
    max_box_height = float('-inf')

    for i, tech in enumerate(techniques):
        group = df[df["technique"] == tech]
        for template in templates:
            data = group[group["template"] == template][metric].dropna()
            if len(data) > 1:
                q1, q3 = np.percentile(data, [25, 75])
                iqr = q3 - q1
                upper_whisker = min(data.max(), q3 + 1.5 * iqr)
                max_box_height = max(max_box_height, upper_whisker)

    spacing = (y_max - y_min) * 0.02
    bar_height = min(max_box_height + spacing, y_max)
    text_height = bar_height + spacing

    for i, tech in enumerate(techniques):
        key = (metric, tech)
        if key in corrected_dict:
            p_raw, p_corr, rej, pos, test_type = corrected_dict[key]

            if rej: # If significant after correction
                ax.plot([pos-0.2, pos+0.2], [bar_height, bar_height], 'k-', linewidth=1.5)

                if p_corr < 0.001:
                    significance = "§"
                elif p_corr < 0.01:
                    significance = "‡"
                else:
                    significance = "†"

                ax.text(pos, text_height, significance,
                        ha="center", va="bottom", fontsize=14, fontname=font_to_use)
                ax.set_ylim(y_min, max_box_height + (y_max - y_min) * 0.1)

    plt.xlabel("Pipeline", fontsize=14, fontname=font_to_use)
    plt.ylabel(metric.replace("_", " ").capitalize(), fontsize=14, fontname=font_to_use)
    plt.xticks(rotation=25, ha='right', fontsize=9, fontname=font_to_use)
    labels = ax.get_xticklabels()
    plt.setp(labels, ha="center", rotation_mode="anchor", position=(0, -0.05))
    '''
    legend = plt.legend(title="Template")
    plt.setp(legend.get_title(), fontname=font_to_use)
    plt.setp(legend.get_texts(), fontname=font_to_use)
    '''
    lg = ax.get_legend()
    if metric == "sparsity":
        if lg is None:
            lg = ax.legend(title="Template")
        else:
            lg.set_title("Template")
        plt.setp(lg.get_title(), fontname=font_to_use)
        plt.setp(lg.get_texts(), fontname=font_to_use)
    else:
        if lg is not None:
            lg.remove()
    
    plt.tight_layout()
    # Saving
    outpath = os.path.join(output_dir, f"{metric}_boxplot.png")
    plt.savefig(outpath, dpi=300, bbox_inches="tight") 
    plt.close()


Usando Wilcoxon per ATR-25-ACT su sparsity con p_raw=0.0000
Usando t-test appaiato per ATR-30-ACT su sparsity con p_raw=0.0000
Usando t-test appaiato per ATR-35-ACT su sparsity con p_raw=0.0000
Usando t-test appaiato per ATR-40-ACT su sparsity con p_raw=0.0000
Usando t-test appaiato per ATR-45-ACT su sparsity con p_raw=0.0000
Usando t-test appaiato per ATR-50-ACT su sparsity con p_raw=0.0000
Usando t-test appaiato per FAST-ACT su sparsity con p_raw=0.0000
Usando t-test appaiato per NOACT su sparsity con p_raw=0.2410
Usando t-test appaiato per ATR-25-ACT su mean_connectivity con p_raw=0.0000
Usando t-test appaiato per ATR-30-ACT su mean_connectivity con p_raw=0.0000
Usando t-test appaiato per ATR-35-ACT su mean_connectivity con p_raw=0.0000
Usando t-test appaiato per ATR-40-ACT su mean_connectivity con p_raw=0.0000
Usando t-test appaiato per ATR-45-ACT su mean_connectivity con p_raw=0.0000
Usando t-test appaiato per ATR-50-ACT su mean_connectivity con p_raw=0.0000
Usando t-test appaiato

In [ ]:
import os, re
from scipy.stats import pearsonr
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

mpl.rcParams["font.family"] = "Arial"
plt.style.use("seaborn-v0_8-whitegrid")
density = False
if density:
    df_conn = pd.read_csv("Y:/share/users/cferritt/DATASET/ADNI3/ANALYSIS/Tractography/Results/connectome_metrics_global.csv")
else:
    df_conn = pd.read_csv("Y:/share/users/cferritt/DATASET/ADNI3/ANALYSIS/Tractography/Results/connectome_no_density_metrics_global.csv")
df_clin = pd.read_excel("Y:/share/users/cferritt/DATASET/ADNI3/Clinical_data/ADNI3_summary.xlsx", sheet_name="Tractography_selected")

# Merge su excel_id / Subject
df = pd.merge(df_conn, df_clin, left_on="excel_id", right_on="Subject")
technique_mapping = {
    "25": "ATR-25-ACT",
    "30": "ATR-30-ACT",
    "35": "ATR-35-ACT",
    "40": "ATR-40-ACT",
    "45": "ATR-45-ACT",
    "50": "ATR-50-ACT",
    "act": "FAST-ACT",
    "no_act": "NOACT"
}
df["technique"] = df["technique"].astype(str).map(technique_mapping).fillna(df["technique"])

metrics = [
    "sparsity", "mean_connectivity", "assortativity", "avg_clustering",
    "global_efficiency", "avg_shortest_path", "transitivity"
]

clinical_vars = ["WMH_WM", "PET_CENTILOID", "MMSE", "MOCA"]
clinical_vars_mapping = {
    "WMH_WM": "WMHB (%)",
    "PET_CENTILOID": "Amyloid PET Centiloid",
    "MMSE": "MMSE Score",
    "MOCA": "MoCA Score"
}

palette = sns.color_palette("tab10", n_colors=len(df["technique"].unique()))
output_dir = r"Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\scatterplots"
os.makedirs(output_dir, exist_ok=True)

for metric in metrics:
    for clinical_var in clinical_vars:
        # nome “carino” per asse/file
        display_clin = clinical_vars_mapping.get(clinical_var, clinical_var)
        safe_display = re.sub(r"[^A-Za-z0-9_-]+", "_", display_clin).strip("_")

        for template in df["template"].unique():
            plt.figure(figsize=(10, 6))
            plt.title(f"{template}", fontsize=20)
            plt.xlabel(display_clin, fontsize=16) 
            plt.ylabel(metric.replace("_", " ").capitalize(),fontsize=16)

            handles, labels = [], []
            techs = sorted(df["technique"].unique())
            for i, tech in enumerate(techs):
                subset = df[(df["template"] == template) & (df["technique"] == tech)]
                x = subset[clinical_var]
                y = subset[metric]
                valid = x.notna() & y.notna()
                x, y = x[valid], y[valid]

                plt.scatter(x, y, color=palette[i], alpha=0.6)

                if len(x) > 1:
                    m, b = np.polyfit(x, y, 1)
                    x_sorted = np.array(sorted(x), dtype=np.float64)
                    line = plt.plot(x_sorted, m * x_sorted + b, color=palette[i], linestyle="--")
                    r, p = pearsonr(x, y)
                    labels.append(f"{tech} (R²={r**2:.2f})")
                    handles.append(line[0])

            if handles:
                plt.legend(handles=handles, labels=labels, title="Pipeline")
            plt.tight_layout()

            filename = f"{metric}_vs_{safe_display}_{template}.png"
            filepath = os.path.join(output_dir, filename)
            plt.savefig(filepath, dpi=300)
            plt.close()


In [ ]:
# ============================================
# ANALISI "UN SOLO LMM" CON PIPELINE=template×technique V2
# - Omnibus clinical:pipeline
# - Pendenze per pipeline con CI
# - Pairwise q-value (heatmap con asterischi)
# - Pearson r per pipeline (descrittivo)
# ============================================
import os, numpy as np, pandas as pd, itertools
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2, norm, pearsonr
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib as mpl
mpl.rcParams["font.family"] = "Arial"
plt.style.use("default")
TITLE_FONTSIZE = 18
AXIS_LABEL_FONTSIZE = 18
TICK_FONTSIZE = 22
ANNOT_FONTSIZE = 20
CBAR_LABEL_FONTSIZE = 18
CBAR_TICK_FONTSIZE = 18
# ============== CONFIG ==============
output_dir = r"Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\PIPELINE_LMM"
os.makedirs(output_dir, exist_ok=True)
os.makedirs(os.path.join(output_dir, "heatmaps"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "slopes"), exist_ok=True)
os.makedirs(os.path.join(output_dir, "pearson"), exist_ok=True)
density=False
if density:
    df_conn = pd.read_csv("Y:/share/users/cferritt/DATASET/ADNI3/ANALYSIS/Tractography/Results/connectome_metrics_global.csv")
else:
    df_conn= pd.read_csv("Y:/share/users/cferritt/DATASET/ADNI3/ANALYSIS/Tractography/Results/connectome_no_density_metrics_global.csv")
df_clin = pd.read_excel("Y:/share/users/cferritt/DATASET/ADNI3/Clinical_data/ADNI3_summary.xlsx", sheet_name="Tractography_selected")

# Merge su excel_id / Subject
df = pd.merge(df_conn, df_clin, left_on="excel_id", right_on="Subject")

# Lista delle metriche da plottare
metrics = [
    "sparsity", "mean_connectivity", "assortativity", "avg_clustering",
    "global_efficiency", "avg_shortest_path", "transitivity"
]
technique_mapping = {
    "25": "ATR-25-ACT",
    "30": "ATR-30-ACT",
    "35": "ATR-35-ACT",
    "40": "ATR-40-ACT",
    "45": "ATR-45-ACT",
    "50": "ATR-50-ACT",
    "act": "FAST-ACT",
    "no_act": "NOACT"
}
templates_mapping = {
    "MNI152NLin2009cAsym": "MNI",
    "MIITRA":"MIITRA"
}
df["template"] = df["template"].astype(str).map(templates_mapping).fillna(df["template"])
df["technique"] = df["technique"].astype(str).map(technique_mapping).fillna(df["technique"])

clinical_vars = [
    "WMH_WM",  
    "APOE4", "PET_CENTILOID",
    "ADAS13", "MMSE", "MOCA",
    "CEREBRUM_TCB", "TOTAL_WHITE"
]
covariates = ["Age", "Sex", "CEREBRUM_TCV"]

rename_map = {}

FALLBACK_OLS = True

# ============== PREP DATI ==============
if rename_map:
    df = df.rename(columns=rename_map)

needed_base = ["Subject", "template", "technique"] + covariates
missing = [c for c in needed_base if c not in df.columns]
if missing:
    raise KeyError(f"Mancano colonne richieste: {missing}")

df["pipeline"] = df["template"].astype(str) + "-" + df["technique"].astype(str)
df["pipeline"] = df["pipeline"].astype("category")
df["template"] = df["template"].astype("category")
df["technique"] = df["technique"].astype("category")
df["Sex"] = df["Sex"].astype("category")

# ============== HELPER ==============
def linear_combo(fe_params, cov, terms, sign=None):
    names = fe_params.index.tolist()
    L = np.zeros(len(names))
    if sign is None:
        sign = [1.0]*len(terms)
    for t, s in zip(terms, sign):
        if t in names:
            L[names.index(t)] += s
    est = float(L @ fe_params.values)
    var = float(L @ cov.values @ L)
    se = np.sqrt(max(var, 0.0))
    z = est / se if se > 0 else np.nan
    p = 2*(1 - norm.cdf(abs(z))) if np.isfinite(z) else np.nan
    zcrit = norm.ppf(0.975)
    low, high = est - zcrit*se, est + zcrit*se
    return est, se, z, p, low, high

def slope_terms_for_pipeline(clinical, level, baseline_level):
    if level == baseline_level:
        return [clinical]
    else:
        return [clinical, f"{clinical}:C(pipeline)[T.{level}]"]

def build_pairwise_q_matrix(fe_params, cov, clinical, pipe_levels):
    baseline = pipe_levels[0]
    pairs, pvals = [], []
    for i, j in itertools.combinations(pipe_levels, 2):
        terms_i = slope_terms_for_pipeline(clinical, i, baseline)
        terms_j = slope_terms_for_pipeline(clinical, j, baseline)
        terms = terms_i + terms_j
        signs = [1.0]*len(terms_i) + [-1.0]*len(terms_j)
        est, se, z, p, low, high = linear_combo(fe_params, cov, terms, signs)
        pairs.append((i, j))
        pvals.append(p)

    pvals = np.array(pvals, dtype=float)
    ok = np.isfinite(pvals)
    qvals = np.full_like(pvals, np.nan)
    if ok.sum() > 0:
        _, q_ok, _, _ = multipletests(pvals[ok], method="fdr_bh")
        qvals[ok] = q_ok

    L = len(pipe_levels)
    qmat = pd.DataFrame(np.nan, index=pipe_levels, columns=pipe_levels)
    for (i, j), q in zip(pairs, qvals):
        qmat.loc[i, j] = q
        qmat.loc[j, i] = q
    np.fill_diagonal(qmat.values, np.nan)
    return qmat

def omnibus_chisq_for_interaction(fe_params, cov, clinical, pipe_levels):
    names = fe_params.index.tolist()
    baseline = pipe_levels[0]
    inter_names = [f"{clinical}:C(pipeline)[T.{lvl}]" for lvl in pipe_levels if lvl != baseline]
    idx = [names.index(nm) for nm in inter_names if nm in names]
    if len(idx) == 0:
        return np.nan, np.nan, 0
    b = fe_params.values[idx]
    S = cov.values[np.ix_(idx, idx)]
    try:
        stat = float(b.T @ np.linalg.inv(S) @ b)
        df = len(idx)
        p = 1 - chi2.cdf(stat, df)
        return stat, p, df
    except np.linalg.LinAlgError:
        return np.nan, np.nan, len(idx)

# funzione per asterischi
def q_to_star(q):
    if q < 0.001:
        return "§"
    elif q < 0.01:
        return "‡"
    elif q < 0.05:
        return "†"
    else:
        return ""

# ============== LOOP PRINCIPALE ==============
for metric in metrics:
    if metric not in df.columns:
        print(f"⚠️ Metrica assente, salto: {metric}")
        continue

    for clinical in clinical_vars:
        if clinical not in df.columns:
            print(f"⚠️ Variabile clinica assente, salto: {clinical}")
            continue

        needed = [metric, clinical, "Subject", "pipeline"] + covariates
        dat = df.dropna(subset=[c for c in needed if c in df.columns]).copy()
        if dat.empty:
            print(f"⚠️ Nessun dato per {metric} ~ {clinical}.")
            continue

        counts = dat.groupby("Subject").size()
        dat = dat[dat["Subject"].isin(counts[counts >= 2].index)].copy()
        if dat["Subject"].nunique() < 2:
            print(f"⚠️ Troppi pochi soggetti con ≥2 osservazioni per {metric} ~ {clinical}.")
            continue

        dat["pipeline"] = dat["pipeline"].astype("category")
        dat["Sex"] = dat["Sex"].astype("category")

        pipe_levels = dat["pipeline"].cat.categories.tolist()
        if len(pipe_levels) < 2:
            print(f"⚠️ Meno di 2 pipeline presenti nei dati per {metric} ~ {clinical}.")
            continue

        formula = f"{metric} ~ {clinical} * C(pipeline) + Age + C(Sex) + CEREBRUM_TCV"

        fit_obj = None
        model_used = None
        try:
            md = smf.mixedlm(formula, data=dat, groups=dat["Subject"])
            fit_obj = md.fit(reml=True, method="lbfgs", maxiter=2000)
            model_used = "MixedLM"
        except Exception as e:
            if FALLBACK_OLS:
                try:
                    ols = smf.ols(formula, data=dat).fit(cov_type="cluster", cov_kwds={"groups": dat["Subject"]})
                    fit_obj = ols
                    model_used = "OLS_cluster"
                    print(f"ℹ️ Fallback OLS cluster per {metric} ~ {clinical}: {e}")
                except Exception as e2:
                    print(f"❌ Fit fallito anche OLS ({metric} ~ {clinical}): {e2}")
                    continue
            else:
                print(f"❌ Fit MixedLM fallito per {metric} ~ {clinical}: {e}")
                continue

        fe = fit_obj.params
        cov = fit_obj.cov_params()

        stat, p_omni, df_omni = omnibus_chisq_for_interaction(fe, cov, clinical, pipe_levels)

        baseline = pipe_levels[0]
        rows = []
        for lvl in pipe_levels:
            terms = slope_terms_for_pipeline(clinical, lvl, baseline)
            est, se, z, p, low, high = linear_combo(fe, cov, terms)
            rows.append({
                "metric": metric, "clinical_var": clinical, "pipeline": lvl,
                "slope": est, "se": se, "z": z, "p": p, "ci_low": low, "ci_high": high,
                "model": model_used, "omnibus_chi2": stat, "omnibus_df": df_omni, "omnibus_p": p_omni,
                "n_rows": dat.shape[0], "n_subjects": dat["Subject"].nunique()
            })
        slopes_df = pd.DataFrame(rows)
        slopes_path = os.path.join(output_dir, "slopes",
                                   f"SLOPES__{metric}__{clinical}.csv".replace(" ", "_"))
        slopes_df.to_csv(slopes_path, index=False)

        qmat = build_pairwise_q_matrix(fe, cov, clinical, pipe_levels)
        qcsv = os.path.join(output_dir, "heatmaps",
                            f"PAIRWISE_Q__{metric}__{clinical}.csv".replace(" ", "_"))
        qmat.to_csv(qcsv)

        # ===== HEATMAP CON ASTERISCHI =====
        annot_mat = qmat.copy().astype(str)
        for i in range(qmat.shape[0]):
            for j in range(qmat.shape[1]):
                val = qmat.iloc[i, j]
                annot_mat.iloc[i, j] = q_to_star(val) if np.isfinite(val) else ""

        mask = np.triu(np.ones_like(qmat, dtype=bool), k=1)

        fig, ax = plt.subplots(figsize=(1.2*len(pipe_levels), 1.0*len(pipe_levels)))
        hm = sns.heatmap(
            qmat.astype(float),
            annot=annot_mat, fmt="",
            cmap="viridis_r", vmin=0, vmax=0.25,
            mask=mask, square=True,
            cbar_kws={"label": "q (BH)", "shrink": 0.9, "pad": 0.02},
            annot_kws={"fontsize": ANNOT_FONTSIZE, "fontweight": "bold"}
        )

        # Titolo e label assi
        #ax.set_title(f"Pairwise q-values slopes\n{metric} ~ {clinical}", fontsize=TITLE_FONTSIZE)
        #ax.set_xlabel("Pipeline", fontsize=AXIS_LABEL_FONTSIZE)
        #ax.set_ylabel("Pipeline", fontsize=AXIS_LABEL_FONTSIZE)

        # Ticks assi
        ax.tick_params(axis="x", labelsize=TICK_FONTSIZE, rotation=90)
        ax.tick_params(axis="y", labelsize=TICK_FONTSIZE, rotation=0)

        # Colorbar: label e tick size
        cbar = hm.collections[0].colorbar
        cbar.ax.tick_params(labelsize=CBAR_TICK_FONTSIZE)
        cbar.set_label("q (BH)", fontsize=CBAR_LABEL_FONTSIZE)

        plt.tight_layout()
        qpng = os.path.join(output_dir, "heatmaps",
                            f"PAIRWISE_Q__{metric}__{clinical}.png".replace(" ", "_"))
        plt.savefig(qpng, dpi=200)
        plt.close()

        corr_rows = []
        for lvl in pipe_levels:
            sub = dat[dat["pipeline"] == lvl]
            x = pd.to_numeric(sub[clinical], errors="coerce")
            y = pd.to_numeric(sub[metric], errors="coerce")
            mask = x.notna() & y.notna()
            x, y = x[mask], y[mask]
            if len(x) >= 3:
                r, p_r = pearsonr(x, y)
                corr_rows.append({"metric": metric, "clinical_var": clinical, "pipeline": lvl,
                                  "r": r, "r2": r**2, "n": len(x), "p_r": p_r})
            else:
                corr_rows.append({"metric": metric, "clinical_var": clinical, "pipeline": lvl,
                                  "r": np.nan, "r2": np.nan, "n": len(x), "p_r": np.nan})
        corr_df = pd.DataFrame(corr_rows)
        corr_path = os.path.join(output_dir, "pearson",
                                 f"PEARSON__{metric}__{clinical}.csv".replace(" ", "_"))
        corr_df.to_csv(corr_path, index=False)

        print(f"✅ {metric} ~ {clinical} | model={model_used} | omnibus p={p_omni:.3g} | "
              f"slopes→ {os.path.basename(slopes_path)} | heatmap→ {os.path.basename(qpng)}")


c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


✅ sparsity ~ WMH_WM | model=MixedLM | omnibus p=0 | slopes→ SLOPES__sparsity__WMH_WM.csv | heatmap→ PAIRWISE_Q__sparsity__WMH_WM.png


c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


✅ sparsity ~ APOE4 | model=MixedLM | omnibus p=1 | slopes→ SLOPES__sparsity__APOE4.csv | heatmap→ PAIRWISE_Q__sparsity__APOE4.png


c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


...

✅ transitivity ~ MOCA | model=MixedLM | omnibus p=0.0057 | slopes→ SLOPES__transitivity__MOCA.csv | heatmap→ PAIRWISE_Q__transitivity__MOCA.png


c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


✅ transitivity ~ CEREBRUM_TCB | model=MixedLM | omnibus p=0 | slopes→ SLOPES__transitivity__CEREBRUM_TCB.csv | heatmap→ PAIRWISE_Q__transitivity__CEREBRUM_TCB.png


c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


✅ transitivity ~ TOTAL_WHITE | model=MixedLM | omnibus p=0 | slopes→ SLOPES__transitivity__TOTAL_WHITE.csv | heatmap→ PAIRWISE_Q__transitivity__TOTAL_WHITE.png


In [ ]:
# ============================================
# RIASSUNTO OMNIBUS (e non solo) SU TUTTE LE METRICHE × VARIABILI CLINICHE
# ============================================
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.multitest import multipletests
mpl.rcParams["font.family"] = "Arial"
plt.style.use("default")
# ========== CONFIG ==========
base_dir   = r"Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\PIPELINE_LMM"
slopes_dir = os.path.join(base_dir, "slopes")
heatm_dir  = os.path.join(base_dir, "heatmaps")
out_dir    = os.path.join(base_dir, "summary")
os.makedirs(out_dir, exist_ok=True)

ALPHA_FDR = 0.05  # soglia per evidenziare significatività FDR

# ========== 1) CARICA SLOPES__*.csv E COSTRUISCI TABELLA OMNIBUS ==========
rows = []
files = sorted(glob.glob(os.path.join(slopes_dir, "SLOPES__*.csv")))
if not files:
    raise FileNotFoundError(f"Nessun file SLOPES__*.csv in {slopes_dir}")

for f in files:
    df = pd.read_csv(f)
    if df.empty: 
        continue
    # un file contiene N righe (una per pipeline) ma stesso omnibus → prendi la prima
    head = df.iloc[0]
    metric  = head["metric"]
    clin    = head["clinical_var"]
    chi2    = head.get("omnibus_chi2", np.nan)
    df_     = head.get("omnibus_df", np.nan)
    p_omni  = head.get("omnibus_p", np.nan)
    model   = head.get("model", "")
    n_rows  = head.get("n_rows", df.shape[0])
    n_subj  = head.get("n_subjects", np.nan)
    n_pipe  = df["pipeline"].nunique() if "pipeline" in df.columns else np.nan

    rows.append({
        "metric": metric,
        "clinical_var": clin,
        "omnibus_chi2": chi2,
        "omnibus_df": df_,
        "omnibus_p": p_omni,
        "model": model,
        "n_rows": n_rows,
        "n_subjects": n_subj,
        "n_pipeline": n_pipe,
        "slopes_path": os.path.basename(f)
    })

omni = pd.DataFrame(rows).sort_values(["metric","clinical_var"]).reset_index(drop=True)

# FDR BH sugli omnibus p (su tutte le coppie metrica–clinica)
omni["q_bh_omnibus"] = np.nan
p = omni["omnibus_p"].to_numpy(dtype=float)
ok = np.isfinite(p)
if ok.sum() > 0:
    _, q, _, _ = multipletests(p[ok], method="fdr_bh")
    omni.loc[ok, "q_bh_omnibus"] = q

# salva tabella long
omni_csv = os.path.join(out_dir, "omnibus_summary.csv")
omni.to_csv(omni_csv, index=False)
print(f"✅ Salvato: {omni_csv}")

# pivot (righe=metriche, colonne=cliniche) con q FDR omnibus
pivot = omni.pivot_table(index="metric", columns="clinical_var", values="q_bh_omnibus", aggfunc="first")
pivot_csv = os.path.join(out_dir, "omnibus_q_matrix.csv")
pivot.to_csv(pivot_csv)
print(f"✅ Salvato: {pivot_csv}")

# opzionale: heatmap veloce della pivot
plt.figure(figsize=(1.2*max(6, pivot.shape[1]), 0.7*max(5, pivot.shape[0])))
sns.heatmap(pivot.astype(float), cmap="viridis_r", vmin=0, vmax=0.25, annot=False, cbar_kws={"label":"q_bh omnibus"})
plt.title("q_bh (FDR) dell’omnibus clinical×pipeline")
plt.tight_layout()
png_path = os.path.join(out_dir, "omnibus_q_heatmap.png")
plt.savefig(png_path, dpi=200); plt.close()
print(f"🖼️ Heatmap: {png_path}")

# ========== 2) “NON SOLO”: ARRICCHISCI CON INDICATORI DI ETEROGENEITÀ ED EFFETTI ==========
# - pairwise_q_frac: quota di coppie pipeline con differenza di pendenza significativa (q<0.05)
# - slope_min/median/max, slope_range
# - sign_consistency: % pipeline con stesso segno della mediana
# - n_sig_slopes_fdr: quante pipeline mostrano slope≠0 a FDR dentro ciascuna famiglia (metrica,clinica)

extra_rows = []
for f in files:
    df = pd.read_csv(f)
    if df.empty:
        continue

    metric = df["metric"].iloc[0]
    clin   = df["clinical_var"].iloc[0]

    # a) pairwise q matrix (se presente)
    qfile = os.path.join(heatm_dir, f"PAIRWISE_Q__{metric}__{clin}.csv".replace(" ", "_"))
    pairwise_q_frac = np.nan
    if os.path.exists(qfile):
        qmat = pd.read_csv(qfile, index_col=0)
        vals = qmat.values[np.triu_indices_from(qmat.values, k=1)]
        vals = vals[np.isfinite(vals)]
        if vals.size > 0:
            pairwise_q_frac = float(np.mean(vals < ALPHA_FDR))

    # b) statistiche sulle pendenze
    slopes = pd.to_numeric(df["slope"], errors="coerce")
    slope_min = float(np.nanmin(slopes)) if slopes.notna().any() else np.nan
    slope_med = float(np.nanmedian(slopes)) if slopes.notna().any() else np.nan
    slope_max = float(np.nanmax(slopes)) if slopes.notna().any() else np.nan
    slope_rng = slope_max - slope_min if np.all(np.isfinite([slope_min, slope_max])) else np.nan

    # coerenza del segno rispetto alla mediana
    sign_med = np.sign(slope_med) if np.isfinite(slope_med) else np.nan
    if np.isfinite(sign_med) and slopes.notna().any():
        sign_consistency = float(np.mean(np.sign(slopes)==sign_med))
    else:
        sign_consistency = np.nan

    # c) pipeline con slope ≠ 0 a FDR all’interno del file
    pvals = pd.to_numeric(df["p"], errors="coerce").to_numpy()
    ok = np.isfinite(pvals)
    n_sig_slopes_fdr = np.nan
    if ok.sum() > 0:
        _, q_s, _, _ = multipletests(pvals[ok], method="fdr_bh")
        n_sig_slopes_fdr = int(np.sum(q_s < ALPHA_FDR))

    extra_rows.append({
        "metric": metric,
        "clinical_var": clin,
        "pairwise_sig_fraction": pairwise_q_frac,
        "slope_min": slope_min,
        "slope_median": slope_med,
        "slope_max": slope_max,
        "slope_range": slope_rng,
        "sign_consistency": sign_consistency,
        "n_sig_slopes_fdr": n_sig_slopes_fdr,
        "n_pipeline": int(df["pipeline"].nunique())
    })

extra = pd.DataFrame(extra_rows).sort_values(["metric","clinical_var"]).reset_index(drop=True)
effects_csv = os.path.join(out_dir, "effects_overview.csv")
extra.to_csv(effects_csv, index=False)
print(f"✅ Salvato: {effects_csv}")

# ========== 3) UNISCI LE INFO IN UN FILE UNICO (facoltativo, comodo per il paper) ==========
merged = pd.merge(omni, extra, on=["metric","clinical_var","n_pipeline"], how="left")
merged_csv = os.path.join(out_dir, "omnibus_plus_effects.csv")
merged.to_csv(merged_csv, index=False)
print(f"✅ Salvato: {merged_csv}")

# Suggerimento di lettura:
# - Ordina merged per q_bh_omnibus crescente → top eterogeneità
# - Guarda pairwise_sig_fraction per capire quanto è “diffusa” la differenza tra pipeline
# - Usa slope_range e sign_consistency per raccontare la direzione/coerenza


✅ Salvato: Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\PIPELINE_LMM\summary\omnibus_summary.csv
✅ Salvato: Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\PIPELINE_LMM\summary\omnibus_q_matrix.csv
🖼️ Heatmap: Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\PIPELINE_LMM\summary\omnibus_q_heatmap.png
✅ Salvato: Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\PIPELINE_LMM\summary\effects_overview.csv
✅ Salvato: Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\PIPELINE_LMM\summary\omnibus_plus_effects.csv


In [2]:
# ============================
# MERGE SLOPES + PEARSON (r, r2, p_r)
# ============================
import os, re, glob
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

# --- CONFIG ---
base_dir    = r"Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\PIPELINE_LMM"
slopes_dir  = os.path.join(base_dir, "slopes")
pearson_dir = os.path.join(base_dir, "pearson")
out_dir     = os.path.join(base_dir, "summary")
os.makedirs(out_dir, exist_ok=True)

# --- Utility per FDR per-gruppo (metric, clinical_var) ---
def add_fdr(group, pcol, qcol):
    p = pd.to_numeric(group[pcol], errors="coerce").to_numpy()
    ok = np.isfinite(p)
    q = np.full_like(p, np.nan, dtype=float)
    if ok.sum() > 0:
        _, q_vals, _, _ = multipletests(p[ok], method="fdr_bh")
        q[ok] = q_vals
    group[qcol] = q
    return group

# --- Utility: metric/clin dal nome file se mancano ---
pat = re.compile(r"__(?P<metric>.+?)__(?P<clin>.+?)\.csv$", flags=re.IGNORECASE)
def parse_metric_clin(path_or_name: str):
    m = pat.search(os.path.basename(path_or_name).replace(" ", "_"))
    if m:
        return m.group("metric").replace("_", " "), m.group("clin").replace("_", " ")
    return None, None

# ========== 1) Carica SLOPES ==========
s_files = sorted(glob.glob(os.path.join(slopes_dir, "SLOPES__*.csv")))
if not s_files:
    raise FileNotFoundError(f"Nessun file SLOPES__*.csv in {slopes_dir}")

slopes_rows = []
for f in s_files:
    df = pd.read_csv(f)
    if df.empty: 
        continue
    if not {"metric","clinical_var"} <= set(df.columns):
        met, clin = parse_metric_clin(f)
        if "metric" not in df.columns: df["metric"] = met
        if "clinical_var" not in df.columns: df["clinical_var"] = clin
    if "pipeline" in df.columns:
        df["pipeline"] = df["pipeline"].astype(str).str.strip()

    # rinomina p delle slopes per evitare conflitti
    if "p" in df.columns:
        df = df.rename(columns={"p": "p_slope"})
    # normalizza eventuali CI
    rename_map = {
        "ci_low":"slope_ci_low","ci_lower":"slope_ci_low",
        "ci_high":"slope_ci_high","ci_upper":"slope_ci_high"
    }
    df = df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns})

    keep_cols = [c for c in [
        "metric","clinical_var","pipeline",
        "slope","slope_ci_low","slope_ci_high","p_slope",
        "model","n_rows","n_subjects"
    ] if c in df.columns]
    slopes_rows.append(df[keep_cols])

slopes_all = pd.concat(slopes_rows, ignore_index=True)

# FDR per p_slope per ciascuna (metric, clinical_var)
if "p_slope" in slopes_all.columns:
    slopes_all = slopes_all.groupby(["metric","clinical_var"], as_index=False)\
                           .apply(add_fdr, pcol="p_slope", qcol="q_slope_bh")\
                           .reset_index(drop=True)

# ========== 2) Carica PEARSON (metric, clinical_var, pipeline, r, r2, n, p_r) ==========
p_files = sorted(glob.glob(os.path.join(pearson_dir, "PEARSON__*.csv")))
pearson_rows = []
for f in p_files:
    df = pd.read_csv(f)
    if df.empty:
        continue
    if not {"metric","clinical_var"} <= set(df.columns):
        met, clin = parse_metric_clin(f)
        if "metric" not in df.columns: df["metric"] = met
        if "clinical_var" not in df.columns: df["clinical_var"] = clin
    if "pipeline" in df.columns:
        df["pipeline"] = df["pipeline"].astype(str).str.strip()

    # rinomina alle colonne standard
    # p_r -> p_pearson ; r -> r_pearson ; mantieni r2 e n
    rename_map = {"r":"r_pearson", "p_r":"p_pearson"}
    df = df.rename(columns=rename_map)

    # tieni solo le colonne attese (se presenti)
    keep_cols = [c for c in [
        "metric","clinical_var","pipeline",
        "r_pearson","r2","n","p_pearson"
    ] if c in df.columns]
    pearson_rows.append(df[keep_cols])

pearson_all = pd.DataFrame()
if pearson_rows:
    pearson_all = pd.concat(pearson_rows, ignore_index=True)

    # FDR per p_pearson per ciascuna (metric, clinical_var)
    if "p_pearson" in pearson_all.columns:
        pearson_all = pearson_all.groupby(["metric","clinical_var"], as_index=False)\
                                 .apply(add_fdr, pcol="p_pearson", qcol="q_pearson_bh")\
                                 .reset_index(drop=True)

# ========== 3) Merge lungo su (metric, clinical_var, pipeline) ==========
if pearson_all.empty:
    merged = slopes_all.copy()
else:
    merged = pd.merge(
        slopes_all, pearson_all,
        on=["metric","clinical_var","pipeline"],
        how="outer",
        validate="m:m"
    )

# Ordina e salva
sort_cols = [c for c in ["metric","clinical_var","pipeline"] if c in merged.columns]
merged = merged.sort_values(sort_cols).reset_index(drop=True)

out_csv = os.path.join(out_dir, "SLOPES_PEARSON__merged_long.csv")
merged.to_csv(out_csv, index=False)
print(f"✅ Salvato: {out_csv}")


C:\Users\cferritt\AppData\Local\Temp\ipykernel_34380\4279428798.py:74: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_fdr, pcol="p_slope", qcol="q_slope_bh")\
C:\Users\cferritt\AppData\Local\Temp\ipykernel_34380\4279428798.py:110: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_fdr, pcol="p_pearson", qcol="q_pearson_bh")\


✅ Salvato: Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\PIPELINE_LMM\summary\SLOPES_PEARSON__merged_long.csv


In [ ]:
import pandas as pd

# Path del file caricato
txt_path = "Y:/share/users/cferritt/DATASET/PARCELLATION/Schaefer/Cortex-Subcortex/Schaefer2018_400Parcels_7Networks_order_Tian_Subcortex_S1_MNI152_label.txt"

# Leggiamo il file riga per riga
with open(txt_path, "r") as f:
    lines = [line.strip() for line in f if line.strip()]

# Ogni due righe: label, poi index+RGBA
records = []
for i in range(0, len(lines), 2):
    label = lines[i]
    parts = lines[i+1].split()
    idx = int(parts[0]) - 1   # 🔑 shift da 1-based → 0-based
    r,g,b,a = map(int, parts[1:])
    records.append((idx, label, r,g,b,a))

df = pd.DataFrame(records, columns=["node","label","R","G","B","A"])

# Parsing della colonna network7
def parse_network(label, node):
    if node < 16:   # primi 16 nodi = Subcortical
        return "Subcortical"
    if "Vis" in label: return "Vis"
    if "SomMot" in label: return "SomMot"
    if "DorsAttn" in label: return "DorsAttn"
    if "SalVentAttn" in label: return "SalVentAttn"
    if "Limbic" in label: return "Limbic"
    if "Cont" in label: return "Cont"
    if "Default" in label: return "Default"
    return "Unknown"

df["network7"] = [parse_network(lab, n) for lab,n in zip(df["label"], df["node"])]

# Salvataggio CSV
csv_path = "Y:/share/users/cferritt/DATASET/PARCELLATION/Schaefer/Cortex-Subcortex/Schaefer2018_400Parcels_7Networks_withSubcortical.csv"
df.to_csv(csv_path, index=False)
print("Salvato:", csv_path)

# Controllo rapido
print("Nodi min:", df["node"].min(), "max:", df["node"].max(), "count:", df["node"].nunique())


Salvato: Y:/share/users/cferritt/DATASET/PARCELLATION/Schaefer/Cortex-Subcortex/Schaefer2018_400Parcels_7Networks_withSubcortical.csv
Nodi min: 0 max: 415 count: 416


In [1]:
# ============================================
# UNICO MODELLO per "template effect" sulle METRICHE NODALI
# Obiettivo: MIITRA − MNI per ogni tecnica (Pipeline), e se varia tra tecniche
# ============================================
import os, itertools, numpy as np, pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from scipy.stats import norm
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick

# ---------- FONT & STILE (configurabili) ----------
FONT_FAMILY   = "Arial"   # es: "Arial", "DejaVu Sans", ...
BASE_FONT_SZ  = 10        # dimensione di base
TITLE_SIZE    = 22        # titoli figure
LABEL_SIZE    = 22       # assi
TICK_SIZE     = 22         # tick axes
LEGEND_SIZE   = 20         # legenda
XTICK_ROT     = 90        # rotazione etichette X per risparmiare spazio

mpl.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.size": BASE_FONT_SZ,
    "axes.titlesize": TITLE_SIZE,
    "axes.labelsize": LABEL_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
    "legend.fontsize": LEGEND_SIZE,
})
plt.style.use("default")

# ---------- PATHS ----------
base_out = r"Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\NODAL_TEMPLATE_EFFECT"
dirs = {
    "root": base_out,
    "estimates": os.path.join(base_out, "estimates"),
    "pairwise_node": os.path.join(base_out, "pairwise_node"),
    "heatmaps": os.path.join(base_out, "heatmaps"),
    "summaries": os.path.join(base_out, "summaries"),
}
for d in dirs.values():
    os.makedirs(d, exist_ok=True)

density = False
if density:
    node_csv = r"Y:/share/users/cferritt/DATASET/ADNI3/ANALYSIS/Tractography/Results/connectome_metrics_local.csv"
else:
    node_csv = r"Y:/share/users/cferritt/DATASET/ADNI3/ANALYSIS/Tractography/Results/connectome_no_density_metrics_local.csv"
clin_xlsx = r"Y:/share/users/cferritt/DATASET/ADNI3/Clinical_data/ADNI3_summary.xlsx"
clin_sheet = "Tractography_selected"

# Yeo-7 mapping
yeo7_csv = r"Y:/share/users/cferritt/DATASET/PARCELLATION/Schaefer/Cortex-Subcortex/Schaefer2018_400Parcels_7Networks_withSubcortical.csv"

# ---------- PARAMS ----------
FALLBACK_OLS = True
ALPHA = 0.001           # soglia per masking/interpretazione
NODES_TO_RUN = None     # None = tutti; es. range(0,100) per test veloce
NET_STRIPE_WIDTH = 0.18 # spessore della stripe colorata a destra della heatmap

# Metriche nodali
metrics = ["degree", "strength", "clustering", "betweenness", "closeness", "local_efficiency"]

# Covariate (come da cella di riferimento)
covariates = ["Age", "Sex", "CEREBRUM_TCV"]

# Mapping nomi pipeline e template
technique_mapping = {
    "25": "ATR-25-ACT", "30": "ATR-30-ACT", "35": "ATR-35-ACT",
    "40": "ATR-40-ACT", "45": "ATR-45-ACT", "50": "ATR-50-ACT",
    "act": "FAST-ACT", "no_act": "NOACT"
}
templates_mapping = {"MNI152NLin2009cAsym": "MNI", "MIITRA": "MIITRA"}

# ---------- COLOR LUT PER NETWORK ----------
net_order = ["Subcortical","Vis","SomMot","DorsAttn",
             "SalVentAttn","Limbic","Cont","Default","Unknown"]
lut = {
    "Subcortical": (0.894, 0.102, 0.110),
    "Vis":         (0.216, 0.494, 0.722),
    "SomMot":      (0.302, 0.686, 0.290),
    "DorsAttn":    (0.596, 0.306, 0.639),
    "SalVentAttn": (1.000, 0.498, 0.0),
    "Limbic":      (1.000, 1.000, 0.200),
    "Cont":        (0.651, 0.337, 0.157),
    "Default":     (0.969, 0.506, 0.749),
    "Unknown":     (0.70, 0.70, 0.70),
}

# ---------- LOAD ----------
df_node = pd.read_csv(node_csv)
df_clin = pd.read_excel(clin_xlsx, sheet_name=clin_sheet)

# Merge robusto
merge_keys_node = [c for c in ["excel_id","Subject","subject","SUBJECT_ID"] if c in df_node.columns]
merge_keys_clin = [c for c in ["Subject","excel_id","subject","SUBJECT_ID"] if c in df_clin.columns]
if not merge_keys_node or not merge_keys_clin:
    raise KeyError("Non trovo una chiave comune per il merge (es. 'excel_id' o 'Subject').")
left_key, right_key = merge_keys_node[0], merge_keys_clin[0]
df = pd.merge(df_node, df_clin, left_on=left_key, right_on=right_key, how="inner")

# Mapping/casting
df["template"]  = df["template"].astype(str).map(templates_mapping).fillna(df["template"].astype(str))
df["technique"] = df["technique"].astype(str).map(technique_mapping).fillna(df["technique"].astype(str))
df["template"]  = pd.Categorical(df["template"], categories=["MNI","MIITRA"], ordered=True)

tech_levels = sorted(df["technique"].astype(str).unique().tolist())
df["technique"] = pd.Categorical(df["technique"], categories=tech_levels, ordered=True)
df["Sex"] = df["Sex"].astype("category")

n_nodes = int(df["node"].nunique())
node_list = sorted(df["node"].unique()) if NODES_TO_RUN is None else list(NODES_TO_RUN)

# ---------- YEO-7 ANNOTATIONS (con auto-detect 0/1-based) ----------
node_ann = pd.read_csv(yeo7_csv)
node_ann_sub = node_ann[["node","network7"]].copy()
node_ann_sub["node"] = node_ann_sub["node"].astype(int)
node_ann_sub = node_ann_sub.drop_duplicates(subset=["node"], keep="first")

eff_nodes_all = pd.Index(df["node"].unique()).astype(int)
ann_nodes_all = pd.Index(node_ann_sub["node"].unique()).astype(int)

# se gli eff_nodes combaciano con ann_nodes-1 ⇒ CSV 1-based → converti a 0-based
if set(eff_nodes_all).issubset(set((ann_nodes_all - 1).tolist())) and 0 in eff_nodes_all:
    node_ann_sub["node"] = node_ann_sub["node"] - 1

# Normalizza possibili maiuscole/minuscole
node_ann_sub["network7"] = node_ann_sub["network7"].replace({"SUBCORTICAL":"Subcortical"})
node_ann_sub["network7"] = node_ann_sub["network7"].fillna("Unknown")

# ---------- HELPERS ----------
def linear_combo(fe_params, cov, terms, signs=None):
    """Costruisce contrasto lineare sui parametri del modello e calcola stima/SE/z/p/CI."""
    names = fe_params.index.tolist()
    L = np.zeros(len(names))
    if signs is None:
        signs = [1.0]*len(terms)
    for t, s in zip(terms, signs):
        if t in names:
            L[names.index(t)] += s
    est = float(L @ fe_params.values)
    var = float(L @ cov.values @ L)
    se  = np.sqrt(max(var, 0.0))
    z   = est / se if se > 0 else np.nan
    p   = 2*(1 - norm.cdf(abs(z))) if np.isfinite(z) else np.nan
    zcrit = norm.ppf(0.975)
    return est, se, z, p, est - zcrit*se, est + zcrit*se

def template_effect_terms_for_tech(level, baseline_tech):
    """Termini per (MIITRA − MNI) alla pipeline 'level'."""
    terms = ["C(template)[T.MIITRA]"]
    if level != baseline_tech:
        terms.append(f"C(template)[T.MIITRA]:C(technique)[T.{level}]")
    return terms

# ---------- MAIN ----------
for metric in metrics:
    if metric not in df.columns:
        print(f"⚠️ Metrica assente: {metric}"); 
        continue

    est_rows      = []
    pairwise_tests = []

    # Dati completi per questa metrica
    needed = ["Subject","node","template","technique","Age","Sex","CEREBRUM_TCV", metric]
    dat_all = df.dropna(subset=[c for c in needed if c in df.columns]).copy()
    if dat_all.empty:
        print(f"⚠️ Nessun dato utile per {metric}")
        continue

    for node in (node_list):
        dat = dat_all[dat_all["node"] == node].copy()
        if dat.empty or dat["Subject"].nunique() < 2:
            continue

        dat["Sex"]      = dat["Sex"].astype("category")
        dat["template"] = pd.Categorical(dat["template"], categories=["MNI","MIITRA"], ordered=True)
        dat["technique"]= pd.Categorical(dat["technique"], categories=tech_levels, ordered=True)

        formula = f"{metric} ~ C(template)*C(technique) + Age + C(Sex) + CEREBRUM_TCV"
        try:
            md = smf.mixedlm(formula, data=dat, groups=dat["Subject"])
            fit_obj = md.fit(reml=True, method="lbfgs", maxiter=2000)
        except Exception:
            if FALLBACK_OLS:
                try:
                    fit_obj = smf.ols(formula, data=dat).fit(
                        cov_type="cluster", cov_kwds={"groups": dat["Subject"]}
                    )
                except Exception:
                    continue
            else:
                continue

        fe, cov = fit_obj.params, fit_obj.cov_params()

        # Effetti template per ogni pipeline
        for lvl in tech_levels:
            terms = template_effect_terms_for_tech(lvl, tech_levels[0])
            est, se, z, p, lo, hi = linear_combo(fe, cov, terms)
            est_rows.append({"metric": metric, "node": node, "technique": lvl,
                             "template_effect": est, "p": p})

        # Pairwise (differenze tra pipeline sull'effetto del template) per questo nodo
        for i, j in itertools.combinations(tech_levels, 2):
            terms_i = template_effect_terms_for_tech(i, tech_levels[0])
            terms_j = template_effect_terms_for_tech(j, tech_levels[0])
            terms   = terms_i + terms_j
            signs   = [1.0]*len(terms_i) + [-1.0]*len(terms_j)
            est, se, z, p, lo, hi = linear_combo(fe, cov, terms, signs)
            pairwise_tests.append({"metric": metric, "node": node, "tech_i": i, "tech_j": j, "p": p})

    # ---- EFFECTS: salvataggio e FDR per metrica ----
    est_df = pd.DataFrame(est_rows)
    if est_df.empty:
        print(f"⚠️ Nessun fit riuscito per {metric}")
        continue
    est_path = os.path.join(dirs["estimates"], f"TEMPLATE_EFFECT__{metric}.csv")
    est_df.to_csv(est_path, index=False)

    # FDR per metrica su tutte le celle nodo×pipeline
    p_mat   = est_df.pivot_table(index="node", columns="technique", values="p", aggfunc="first")
    eff_mat = est_df.pivot_table(index="node", columns="technique", values="template_effect", aggfunc="first")

    pvals_flat = p_mat.values.flatten()
    mask = np.isfinite(pvals_flat)
    q_flat = np.full_like(pvals_flat, np.nan, dtype=float)
    if mask.sum() > 0:
        _, q_ok, _, _ = multipletests(pvals_flat[mask], method="fdr_bh")
        q_flat[mask] = q_ok
    q_mat = pd.DataFrame(q_flat.reshape(p_mat.shape), index=p_mat.index, columns=p_mat.columns)

    # --- HEATMAP (masked) con stripe di rete a destra ---
    eff_masked = eff_mat.copy()
    eff_masked[q_mat >= ALPHA] = 0  # masking (NOTA: 0 ≠ effetto nullo stimato; solo non significativo)

    eff_masked_withnet = (
        eff_masked.reset_index()
                  .rename(columns={"index":"node"})
                  .merge(node_ann_sub, on="node", how="left")
    )
    eff_masked_withnet["network7"] = eff_masked_withnet["network7"].replace({"SUBCORTICAL":"Subcortical"})
    eff_masked_withnet["network7"] = eff_masked_withnet["network7"].fillna("Unknown")

    # Matrice da plottare nell'ordine naturale dei nodi
    mat = eff_masked_withnet.set_index("node").drop(columns="network7")
    # Colori rete allineati esattamente all'ordine dei nodi in 'mat'
    row_net = eff_masked_withnet.set_index("node").loc[mat.index, "network7"]
    row_colors = row_net.map(lambda k: lut.get(k, lut["Unknown"])).tolist()

    # Limiti colorbar basati sulla matrice non mascherata
    vmax = np.nanpercentile(np.abs(eff_mat.values), 95)
    vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1.0
    vmin = -vmax

    fig, ax = plt.subplots(figsize=(12, 8))
    ax = sns.heatmap(
    mat,
    cmap="RdBu_r",
    center=0, vmin=vmin, vmax=vmax,
    cbar_kws={
        "label": "Template effect (MIITRA − MNI)",
        "format": mtick.FormatStrFormatter('%.1e')  # notazione scientifica con 2 decimali
    }, yticklabels=False)
    # Stripe rete: attaccata al bordo destro
    ncols = mat.shape[1]
    for y, color in enumerate(row_colors):
        ax.add_patch(plt.Rectangle((ncols, y), NET_STRIPE_WIDTH, 1, color=color, lw=0, clip_on=False))

    # Etichette e rotazione asse X
    #ax.set_title(f"{metric}")
    ax.set_xlabel("Pipeline"); ax.set_ylabel("Node")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=XTICK_ROT, ha="right")

    # Legenda reti in basso (titolo richiesto)
    handles = [mpatches.Patch(color=lut[n], label=n)
               for n in net_order if n in row_net.unique()]
    ax.legend(handles=handles, bbox_to_anchor=(0.5, -0.35), loc='upper center',
              title="Yeo7 Networks + Subcortical",ncol=8, frameon=False)

    plt.tight_layout()
    heat_path = os.path.join(dirs["heatmaps"], f"HEATMAP_TEMPLATE_EFFECT_MASKED__{metric}_netColor.png")
    plt.savefig(heat_path, dpi=220, bbox_inches="tight")
    plt.close(fig)

    # ---- PAIRWISE: % nodi con differenza tra pipeline (q < ALPHA) ----
    pairwise_df = pd.DataFrame(pairwise_tests)
    if not pairwise_df.empty:
        # FDR su tutte le entry pairwise di questa metrica (tutti i nodi × coppie)
        ok = pairwise_df["p"].notna()
        qvals = np.full(len(pairwise_df), np.nan)
        if ok.sum() > 0:
            _, q_ok, _, _ = multipletests(pairwise_df.loc[ok, "p"], method="fdr_bh")
            qvals[ok] = q_ok
        pairwise_df["q"] = qvals

        pipes = tech_levels
        M = pd.DataFrame(np.nan, index=pipes, columns=pipes, dtype=float)
        for (i, j), sub in pairwise_df.groupby(["tech_i","tech_j"]):
            arr = (sub["q"] < ALPHA).to_numpy()
            perc = 100.0 * arr.sum() / max(1, arr.size)
            M.loc[i, j] = M.loc[j, i] = perc

        plt.figure(figsize=(1.1*len(pipes), 1.0*len(pipes)))
        upper_mask = np.triu(np.ones_like(M, dtype=bool), k=1)
        ax = sns.heatmap(M, cmap="viridis", vmin=0, vmax=100, mask=upper_mask,
                         annot=True, fmt=".0f",
                         cbar_kws={"label": f"% nodes with differences in template effect (q<{ALPHA})"})
        ax.set_xlabel("Pipeline"); ax.set_ylabel("Pipeline")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=XTICK_ROT, ha="right")
        ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
        plt.tight_layout()
        pair_path = os.path.join(dirs["summaries"], f"PERCENT_NODES_DIFF_TEMPLATE_EFFECT__{metric}.png")
        plt.savefig(pair_path, dpi=220)
        plt.close()

    # ---- NETWORK SUMMARY (CSV + barplot) ----
    tmp = est_df.merge(node_ann_sub, on="node", how="left")
    q_long = q_mat.stack().reset_index().rename(columns={0:"q","level_0":"node","level_1":"technique"})
    tmp = tmp.merge(q_long, on=["node","technique"], how="left")
    tmp = tmp.dropna(subset=["network7","template_effect"])

    net_sum = (tmp.groupby(["network7","technique"], as_index=False)
                 .agg(mean_effect=("template_effect","mean"),
                      med_effect=("template_effect","median"),
                      n_nodes=("node","nunique"),
                      pct_sig=("q", lambda x: 100*np.mean(np.array(x)<ALPHA) if x.notna().sum()>0 else np.nan))
             )
    net_csv = os.path.join(dirs["summaries"], f"NETWORK_SUMMARY__{metric}.csv")
    net_sum.to_csv(net_csv, index=False)

    plt.figure(figsize=(12,6))
    ax = sns.barplot(data=net_sum, x="technique", y="pct_sig", hue="network7", palette=lut)
    ax.set_xlabel("Pipeline")
    ax.set_ylabel(f"% nodes with q<{ALPHA}")
    #ax.set_title(f"{metric}")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=XTICK_ROT, ha="right")
    # titolo legenda richiesto
    #leg = ax.legend(title="Yeo7 Networks + Subcortical", frameon=False, ncol=3)
    # Togli la legenda
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()
    plt.tight_layout()
    net_fig = os.path.join(dirs["summaries"], f"NETWORK_PCTSIG__{metric}.png")
    plt.savefig(net_fig, dpi=200)
    plt.close()

    print(f"✅ {metric} → effects: {os.path.basename(est_path)} | heatmap: {os.path.basename(heat_path)}")


C:\Users\cferritt\AppData\Local\Temp\ipykernel_34380\662494834.py:94: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_node = pd.read_csv(node_csv)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
C:\Users\cferritt\AppData\Local\Temp\ipykernel_34380\662494834.py:341: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.


✅ degree → effects: TEMPLATE_EFFECT__degree.csv | heatmap: HEATMAP_TEMPLATE_EFFECT_MASKED__degree_netColor.png


c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
C:\Users\cferritt\AppData\Local\Temp\ipykernel_34380\662494834.py:341: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.

✅ strength → effects: TEMPLATE_EFFECT__strength.csv | heatmap: HEATMAP_TEMPLATE_EFFECT_MASKED__strength_netColor.png


c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_mode

...

✅ betweenness → effects: TEMPLATE_EFFECT__betweenness.csv | heatmap: HEATMAP_TEMPLATE_EFFECT_MASKED__betweenness_netColor.png


C:\Users\cferritt\AppData\Local\Temp\ipykernel_34380\662494834.py:341: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=XTICK_ROT, ha="right")


✅ closeness → effects: TEMPLATE_EFFECT__closeness.csv | heatmap: HEATMAP_TEMPLATE_EFFECT_MASKED__closeness_netColor.png


c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Users\cferritt\AppData\Local\anaconda3\envs\snowflakes\Lib\site-packages\statsmodels\regression\mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)
c:\Us

✅ local_efficiency → effects: TEMPLATE_EFFECT__local_efficiency.csv | heatmap: HEATMAP_TEMPLATE_EFFECT_MASKED__local_efficiency_netColor.png


In [ ]:
import pandas as pd
import numpy as np
import os

# Percorsi (da adattare se servono)
pvalues_file = "pvalues_corrected_global.csv"
omnibus_file = os.path.join(
    r"Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\PIPELINE_LMM\summary",
    "omnibus_summary.csv"
)
output_dir = r"Y:\share\users\cferritt\DATASET\ADNI3\ANALYSIS\Tractography\Results\SummaryTables"
os.makedirs(output_dir, exist_ok=True)

# === Table 1: paired tests MIITRA vs MNI ===
if os.path.exists(pvalues_file):
    df_pvals = pd.read_csv(pvalues_file)
    # pivot: righe = metric, colonne = technique, valore = q-value (più compatto)
    table1 = df_pvals.pivot_table(index="metric", columns="technique", values="p_corr", aggfunc="first")
    # Aggiungiamo asterischi per significatività
    def starify(q):
        if pd.isna(q):
            return ""
        if q < 0.001:
            return f"{q:.3g}§"
        elif q < 0.01:
            return f"{q:.3g}‡"
        elif q < 0.05:
            return f"{q:.3g}†"
        else:
            return f"{q:.3g}"
    table1_fmt = table1.applymap(starify)
else:
    table1_fmt = pd.DataFrame()

# === Table 2: omnibus tests clinical × pipeline ===
if os.path.exists(omnibus_file):
    df_omni = pd.read_csv(omnibus_file)
    # pivot: righe = metric, colonne = clinical_var, valore = q_bh_omnibus
    table2 = df_omni.pivot_table(index="metric", columns="clinical_var", values="q_bh_omnibus", aggfunc="first")
    # Aggiungiamo asterischi anche qui
    table2_fmt = table2.applymap(starify)
else:
    table2_fmt = pd.DataFrame()
# Stampa le tabelle in console
print("\n=== Table 1: Paired tests (MIITRA vs MNI) ===")
print(table1_fmt)

print("\n=== Table 2: Omnibus tests (clinical × pipeline) ===")
print(table2_fmt)

# Esporta in Excel
out_path = os.path.join(output_dir, "results_tables.xlsx")
with pd.ExcelWriter(out_path) as writer:
    table1_fmt.to_excel(writer, sheet_name="Paired_tests")
    table2_fmt.to_excel(writer, sheet_name="Omnibus_tests")

print(f"Tabelle salvate in: {out_path}")


C:\Users\cferritt\AppData\Local\Temp\ipykernel_20496\731063592.py:31: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  table1_fmt = table1.applymap(starify)
C:\Users\cferritt\AppData\Local\Temp\ipykernel_20496\731063592.py:41: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  table2_fmt = table2.applymap(starify)



=== Table 1: Paired tests (MIITRA vs MNI) ===
technique         ATROPOS-25-ACT ATROPOS-30-ACT ATROPOS-35-ACT ATROPOS-40-ACT  \
metric                                                                          
assortativity        1.29e-09***    7.26e-12***    2.48e-12***    2.98e-10***   
avg_clustering             0.324           0.93           0.37        0.0242*   
avg_shortest_path    2.02e-08***    9.56e-09***    3.42e-10***     4.3e-10***   
global_efficiency    2.49e-12***    1.66e-13***    6.59e-14***    9.18e-15***   
mean_connectivity    1.09e-14***    4.06e-17***    1.95e-16***    9.94e-16***   
sparsity             1.31e-10***    3.62e-15***    6.59e-14***    1.05e-13***   
transitivity         4.19e-14***    4.12e-14***    1.76e-12***     2.9e-14***   

technique         ATROPOS-45-ACT ATROPOS-50-ACT     FAST-ACT        NOACT  
metric                                                                     
assortativity        2.62e-09***    3.27e-07***  0.000239***        0.6